In [0]:
%run "../common/config"

In [0]:
# Databricks notebook source
# MAGIC %run "../common/config"

# COMMAND ----------
from pyspark.sql import functions as F

etf_df = (
    spark.read.option("inferSchema", True).option("header", True)
    .excel(f"{MASTERS_VOLUME}/Etf_List.xlsx")
    .toDF("Symbol", "Name")
)
etf_df = etf_df.withColumn("Symbol", F.trim(F.col("Symbol"))) \
                .withColumn("Name", F.trim(F.col("Name")))
etf_df = etf_df.subtract(etf_df.limit(1))  # drop stray header-row artifact

etf_df = etf_df.withColumn(
    "Asset_Type",
    F.when(F.col("Name").rlike("(?i)gold|commodity|silver"), "Commodity")
     .when(F.col("Name").rlike("(?i)bond|gilt|debt|government securities|treasury|money market|liquid"), "Bond")
     .otherwise("Equity")
)

etf_df = etf_df.withColumn(
    "Sub_Type",
    F.when(F.col("Name").rlike("(?i)gold"), "Gold")
     .when(F.col("Name").rlike("(?i)silver"), "Silver")
     .when(F.col("Name").rlike("(?i)liquid|overnight"), "Liquid")
     .when(F.col("Name").rlike("(?i)gilt|government|treasury"), "Gilt")
     .when(F.col("Name").rlike("(?i)bharat bond"), "Target Maturity")
     .when(F.col("Name").rlike("(?i)sensex|nifty|nasdaq|index|bse"), "Index")
     .when(F.col("Name").rlike("(?i)defence|health|bank|pharma|it|sector"), "Sector")
     .when(F.col("Name").rlike("(?i)value|momentum|quality|low volatility|beta"), "Factor")
     .when(F.col("Name").rlike("(?i)cpse|psu"), "PSU")
     .when(F.col("Name").rlike("(?i)shariah"), "Thematic")
     .otherwise(None)
)

etf_df = etf_df.withColumn("ingestion_timestamp", F.current_timestamp())
etf_df.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.etf_master_raw")
display(etf_df)

Symbol,Name,Asset_Type,Sub_Type,ingestion_timestamp
SILVER360,Commodity-Silver,Commodity,Silver,2026-09-01T11:11:24.177Z
ECAPINSURE,BSE Capital Markets & Insurance Total Return Index,Equity,Index,2026-09-01T11:11:24.177Z
MOM30IETF,ICICI Prudential Nifty 200 Momentum 30 ETF,Equity,Index,2026-09-01T11:11:24.177Z
MOCAPITAL,Nifty Capital Market Total Return Index,Equity,Index,2026-09-01T11:11:24.177Z
AONENIFTY,Nifty 50 Index,Equity,Index,2026-09-01T11:11:24.177Z
GOLDADD,DSP Gold ETF,Commodity,Gold,2026-09-01T11:11:24.177Z
HDFCMID150,HDFC NIFTY Midcap 150 ETF,Equity,Index,2026-09-01T11:11:24.177Z
ABSLNN50ET,Nifty Next 50,Equity,Index,2026-09-01T11:11:24.177Z
GOLD1,Gold,Commodity,Gold,2026-09-01T11:11:24.177Z
AONEGOLD,Domestic price of Gold,Commodity,Gold,2026-09-01T11:11:24.177Z
